In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(verbose=True)

model_deepseek = init_chat_model(
    model='deepseek-v4-pro',
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {
        "thinking" :{
            "type":"disabled"
        }
    }
)

# JSON Schema格式的使用

举例1：简单返回结构

In [5]:
json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
 }

structured_model = model_deepseek.with_structured_output(json_schema,method="json_schema")
response = structured_model.invoke("给出盗梦空间的信息")

print(response)
print(type(response))


{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}
<class 'dict'>


举例2：

In [7]:
"""
使用 JSON Schema 定义嵌套结构
"""
# 1. 定义嵌套的 JSON Schema
project_schema = {
    "title": "MovieInfo",
    "description": "包含电影标题、上映年份、导演、演员和评分的电影对象",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影标题"},
        "year": {"type": "integer", "description": "上映年份"},
        "director": {"type": "string", "description": "导演"},
        "cast": {  # 定义嵌套数组
            "type": "array",
            "description": "演员列表",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "演员姓名"},
                    "role": {"type": "string", "description": "演员角色"}
                },
                "required": ["name", "role"]
            }
        },
        "rating": {"type": "number", "description": "评分（10分制）"}
    },
    "required": ["title", "year", "director", "cast", "rating"]
}
# 绑定 JSON Schema 到模型
structured_model = model_deepseek.with_structured_output(project_schema,method="json_schema")
# 调用模型
response = structured_model.invoke("生成一个关于《星际穿越》的电影信息，包含导演、演员、评分")
print(response)


{'title': '星际穿越', 'year': 2014, 'director': '克里斯托弗·诺兰', 'cast': [{'name': '马修·麦康纳', 'role': '库珀'}, {'name': '安妮·海瑟薇', 'role': '布兰德'}, {'name': '杰西卡·查斯坦', 'role': '墨菲'}, {'name': '迈克尔·凯恩', 'role': '布兰德教授'}], 'rating': 9.4}
